# 3. OpenAI Debugging Tutor
######
In this notebook, we will build a simple debugging tutor using the OpenAI API.

## Learning goals
- Load an API key from a shared `.env` file
- Create a simple client with the OpenAI Python package
- Use a short Socratic tutor prompt
- Launch a Gradio chat interface

This notebook mirrors the local version so students can compare a hosted model with a small local model.


## 1. Import the packages

We need:
- `os` for environment variables
- `load_dotenv` to read the shared `.env` file
- `OpenAI` to call the API
- `gradio` to build a simple chat interface


In [8]:
import os

try:
    from dotenv import load_dotenv
except ImportError:
    %pip install python-dotenv
    from dotenv import load_dotenv

try:
    from openai import OpenAI
except ImportError:
    %pip install openai
    from openai import OpenAI

try:
    import gradio as gr
except ImportError:
    %pip install gradio
    import gradio as gr


## 2. Load the API key

The API key should not be written directly in the notebook.
Instead, we load it from a shared `.env` file.

If your class uses a different path, update the file path below.


In [9]:
load_dotenv('/home/jovyan/shared/.env')

openai_api_key = os.getenv('OPENAI_API_KEY') or os.getenv('openai_API_KEY')
print('API key loaded:', '✅' if openai_api_key else '❌ not found')


API key loaded: ✅


## 3. Create the client

Now we create an OpenAI client and choose a model name.
If your project uses a different model, you can change the `model_name` variable.


In [10]:
client = OpenAI(api_key=openai_api_key)
model_name = 'gpt-4o-mini'
print('Using model:', model_name)


Using model: gpt-4o-mini


## 4. Write the tutor prompt

We use the same short Socratic prompt as the local notebook.
This keeps the comparison focused on the model, not on different instructions.


In [11]:
system_prompt = """
You are a Socratic debugging tutor.

Rules:
- Do not give answers or corrected code.
- Ask one guiding question at a time.
- Give only one small hint or check.
- Keep it short and clear.
- Focus on one issue at a time.
- End with one short question.
""".strip()


## 5. Define the chat function

To keep this notebook simple, we turn the chat history into plain text and send it with the new message.
The model then returns a short tutoring response.


In [12]:
def chat_with_openai(message, history):
    transcript_parts = []

    for user_msg, assistant_msg in history:
        if user_msg:
            transcript_parts.append(f"Student: {user_msg}")
        if assistant_msg:
            transcript_parts.append(f"Tutor: {assistant_msg}")

    transcript = "\n\n".join(transcript_parts)

    if transcript:
        user_input = f"Conversation so far:\n{transcript}\n\nNew student message:\n{message}"
    else:
        user_input = message

    response = client.responses.create(
        model=model_name,
        instructions=system_prompt,
        input=user_input,
        max_output_tokens=120,
    )

    return response.output_text.strip()

## 6. Launch the Gradio app

Now we wrap the tutor in a Gradio interface.
This makes it easy to compare the hosted model with the local notebook.


In [13]:
demo = gr.ChatInterface(
    fn=chat_with_openai,
    title="OpenAI Socratic Debugging Tutor",
    description=f"Model: {model_name}",
    examples=[
        "My Python loop only prints the first item. What should I check?",
        "I get an IndexError in my list code. Can you help me debug it?",
        "My function returns None when I expect a number.",
    ],
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://e8542d76a0d0fe4a0e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Reflection questions

- How does the OpenAI tutor compare with the local tutor?
- Does the hosted model ask better questions, or just longer ones?
- What are the trade-offs between a local notebook and an API notebook?
